# DSC 670 Term Project - Stock Analysis with Financial APIs and GenAI API 
## Zicheng (Chance) Xu
## Professor: Frank Neugebauer
## Bellevue University

## Project Statement
\<Placeholder\>

## Thought Process
For this project, since it's about GenAI and incorporating prompt is expected when it comes to GenAI, I will leverage the power of GenAI for charting analysis, along with technical indicators, to derive some buy/hold/sell ratings on the list of stocks I am actively watching that covers multiple sectors and industries with a market cap range from small caps to mega caps.

Here's the pipeline of the project planned:
- reads the CSV (with symbols like SMTC.O) exported from investing.com and normalizes them for Yahoo Finance,
- pulls free daily OHLCV via yfinance,
- computes RSI, MACD, Bollinger, %B, Bandwidth, 52-week Hi/Lo, lightweight Support/Resistance, and basic TD Sequential,
- calls OpenAI to produce a GenAI chart analysis (compact, 1–3 sentences) and a structured stance/score,
- blends the LLM score with local indicators into a transparent Buy/Hold/Sell,
- fine tune the model to function the same way the prompt engineered model behaves,
- uses Streamlit to create a web application that uses my solution with search, sort, quick filters, colored badges, and a “GenAI Analysis” column before the final call.

## Import and Configuration
For this section, we will set file paths (ticker watchlist CSV), lookback window (As needed by stock indicators), OpenAI model (gpt-4o-mini for cost efficiency), and score thresholds. We will implement flexibility so we can toggle whether all tickers get GenAI summaries or only “interesting” ones to control spend.

In [5]:
import os, re, time, json
from datetime import datetime, timedelta
from typing import List, Dict, Tuple
from dotenv import load_dotenv

Now let's try to import dependencies and install missing packages.

In [7]:
try:
    import yfinance as yf
except ImportError:
    !pip -q install yfinance
    import yfinance as yf

import pandas as pd
import numpy as np

# OpenAI (needed for GenAI analysis)
try:
    from openai import OpenAI
except Exception:
    !pip -q install "openai>=1.35.0"
    from openai import OpenAI

Now let's define file paths, lookback windows, model choice, scoring thresholds, and knobs to control cost / behavior.

In [9]:
WATCHLIST_CSV = "Chance_Watchlist.csv"   
TICKER_COL = "Symbol"    # We might enable auto-detect just in case it changes in the future
LOOKBACK_DAYS = 420      # ~20 months of daily bars
INTERVAL = "1d"          # Daily chart is the norm and we will start from here
BATCH_SIZE = 25          # polite batch size for yfinance
HTML_OUT = f"./watchlist_dashboard_{datetime.now().strftime('%Y%m%d')}.html"
CSV_OUT  = f"./watchlist_metrics_{datetime.now().strftime('%Y%m%d')}.csv"

I've created the .env file with the OpenAI API key in the current working folder so we can move on to set the model, and extract the key.

In [11]:
OPENAI_MODEL = "gpt-4o-mini"  # We can always upgrade to newer models for better performance
LLM_TEMPERATURE = 0.2

load_dotenv()  # read the .env file in the same folder
api_key = os.getenv("OPENAI_API_KEY")

print("Key loaded?", bool(api_key))  # We should expect it to print True

Key loaded? True


To cut costs, we can limit LLM calls to "interesting" names only. For now we will summarize all 129 tickers in my watchlist. We can set False to only summarize filtered tickers (see filter rule below).

In [13]:
SUMMARIZE_ALL_TICKERS = True
SLEEP_BETWEEN_CALLS_SEC = 0.05

I will set the threshold to +2 for BUY and –2 for SELL because this creates a balanced buffer that filters out weak or conflicting signals, ensuring that only strong bullish or bearish conditions trigger a clear trading stance. A final score of +2 or higher reflects multiple reinforcing bullish indicators (e.g., MACD, RSI, or TD Sequential alignment), while –2 or lower captures strong downside momentum or overbought exhaustion. Scores that fall between –2 and +2 are classified as HOLD, representing neutral or mixed market conditions where taking no action is statistically safer.

In [15]:
SCORE_THRESH_BUY  = +2   # >= +2 => BUY
SCORE_THRESH_SELL = -2   # <= -2 => SELL

## Read & Normalize Tickers
In this section we:
- Load the CSV and auto-detect the ticker column if not provided.
- Normalize symbols for Yahoo Finance:
    * Exchange suffixes like ".O", ".N", ".K", ".AS" that I found in the exported file are removed (e.g., "SMTC.O" -> "SMTC").
    * Share-class dots are converted to dashes, e.g., "BRK.B" -> "BRK-B", "HEI.A" -> "HEI-A".

In [17]:
EXCHANGE_SUFFIXES = {"O","N","K","AS","L","T","KS","KQ","SW","PA","DE","MI","BR","TO","V"}

def normalize_symbol_for_yahoo(sym: str) -> str:
    s = sym.strip().upper()
    # If the symbol contains a dot, decide whether it's an exchange code or a share-class suffix
    if "." in s:
        base, suffix = s.rsplit(".", 1)
        # If suffix looks like an exchange code, drop it entirely.
        if suffix in EXCHANGE_SUFFIXES:
            return base
        # If single-letter suffix (common for US share classes), convert to dash.
        if len(suffix) == 1 and suffix.isalpha():
            return f"{base}-{suffix}"
        # Fallback: remove the suffix (works for many Reuters-style codes)
        return base
    # Special fixes
    if s in {"BRK.B"}: return "BRK-B"
    if s in {"BRK.A"}: return "BRK-A"
    return s

def load_watchlist(csv_path: str, ticker_col: str = None) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    # Try to infer the ticker column if not provided
    if ticker_col and ticker_col in df.columns:
        col = ticker_col
    else:
        candidates = [c for c in df.columns if df[c].astype(str).str.match(r"^[A-Za-z0-9.\-]+$").mean() > 0.5]
        col = candidates[0] if candidates else df.columns[0]
    df = df[[col]].copy()
    df.columns = ["symbol_original"]
    df["symbol"] = df["symbol_original"].astype(str).apply(normalize_symbol_for_yahoo)
    # Drop empties and de-duplicate
    df = df[df["symbol"].str.len() > 0].drop_duplicates(subset=["symbol"]).reset_index(drop=True)
    return df

In [18]:
watch_df = load_watchlist(WATCHLIST_CSV, TICKER_COL)
tickers = watch_df["symbol"].tolist()
print(f"Loaded {len(tickers)} normalized tickers. Sample:", tickers[:10])

Loaded 129 normalized tickers. Sample: ['MXL', 'LAC', 'SMTC', 'CLSK', 'MARA', 'RIOT', 'HOOD', 'IREN', 'WULF', 'KSCP']


The tickers are loaded successfully. We are good to move on to the yahoo finance api to derive the technicals.

## Download OHLCV
Let's now fetch daily candles for the lookback window in batches to reduce throttling, returning a dict of per-ticker DataFrames.

In [21]:
def fetch_prices_yf(tickers: List[str], start: str, end: str, interval: str = "1d") -> Dict[str, pd.DataFrame]:
    data = {}
    for i in range(0, len(tickers), BATCH_SIZE):
        batch = tickers[i:i+BATCH_SIZE]
        try:
            df = yf.download(
                tickers=" ".join(batch), start=start, end=end, interval=interval,
                group_by="ticker", auto_adjust=False, threads=True, progress=False
            )
        except Exception as e:
            print(f"[WARN] batch {i}-{i+len(batch)} failed: {e}")
            df = pd.DataFrame()

        if isinstance(df.columns, pd.MultiIndex):  # multi-ticker frame
            for t in batch:
                if t in df.columns.levels[0]:
                    sub = df[t].copy()
                    sub.columns = [c.capitalize() for c in sub.columns]  # Open, High, Low, Close, Adj Close, Volume
                    sub = sub.dropna(how="all")
                    if not sub.empty:
                        data[t] = sub
        else:  # single ticker fallback
            sub = df.copy()
            if not sub.empty:
                sub.columns = [c.capitalize() for c in sub.columns]
                data[batch[0]] = sub

        time.sleep(0.2)  # polite pause
    return data

In [22]:
from datetime import timezone
end_date = datetime.now(timezone.utc).date()
start_date = end_date - timedelta(days=LOOKBACK_DAYS)
price_data = fetch_prices_yf(
    tickers,
    start=str(start_date),
    end=str(end_date + timedelta(days=1)),
    interval=INTERVAL
)
print(f"Fetched price history for {len(price_data)}/{len(tickers)} tickers.")

Fetched price history for 129/129 tickers.


Now that we have the base data of the 129 tickers, we will move on to derive indicators.

## Local indicator functions
In this section we implement RSI, MACD, Bollinger Bands, %B, bandwidth, 52-week high/low, a lightweight support/resistance, and a basic TD Sequential. We can add/remove some later but for now we will focus on these indicators.

In [25]:
def ema(series: pd.Series, span: int) -> pd.Series:
    return series.ewm(span=span, adjust=False).mean()

def rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    up = pd.Series(np.where(delta > 0, delta, 0.0), index=close.index)
    dn = pd.Series(np.where(delta < 0, -delta, 0.0), index=close.index)
    roll_up = up.rolling(period).mean()
    roll_dn = dn.rolling(period).mean()
    rs = roll_up / roll_dn.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def macd(close: pd.Series, fast=12, slow=26, signal=9):
    macd_line = ema(close, fast) - ema(close, slow)
    signal_line = ema(macd_line, signal)
    hist = macd_line - signal_line
    return macd_line, signal_line, hist

def bollinger(close: pd.Series, period=20, num_std=2.0):
    ma = close.rolling(period).mean()
    sd = close.rolling(period).std(ddof=0)
    upper = ma + num_std * sd
    lower = ma - num_std * sd
    pctb = (close - lower) / (upper - lower)
    bandwidth = (upper - lower) / ma
    return ma, upper, lower, pctb, bandwidth

def fifty_two_week_metrics(close: pd.Series) -> Tuple[float, float]:
    last_252 = close.tail(252)
    return float(last_252.max()), float(last_252.min())

def local_extrema_support_resistance(close: pd.Series, lookback: int = 180) -> Tuple[float, float]:
    s = close.tail(lookback)
    last_px = s.iloc[-1]
    mins = s[(s.shift(1) > s) & (s.shift(-1) > s)]
    maxs = s[(s.shift(1) < s) & (s.shift(-1) < s)]
    support = mins[mins < last_px].max() if not mins.empty else np.nan
    resistance = maxs[maxs > last_px].min() if not maxs.empty else np.nan
    return (float(support) if pd.notna(support) else np.nan,
            float(resistance) if pd.notna(resistance) else np.nan)

def td_sequential_basic(df: pd.DataFrame) -> Dict[str, object]:
    close = df["Close"]; high = df["High"]; low = df["Low"]
    n = len(df)
    buy_count = sell_count = 0
    last_buy9_idx = last_sell9_idx = None
    buy_perfected = sell_perfected = False

    for i in range(n):
        if i >= 4:
            if close.iloc[i] < close.iloc[i-4]:
                buy_count += 1; sell_count = 0
            elif close.iloc[i] > close.iloc[i-4]:
                sell_count += 1; buy_count = 0
            else:
                buy_count = sell_count = 0

        if buy_count == 9:
            last_buy9_idx = i
            if i >= 8:
                l8, l9 = low.iloc[i-1], low.iloc[i]
                l6, l7 = low.iloc[i-3], low.iloc[i-2]
                buy_perfected = (l8 <= min(l6, l7)) or (l9 <= min(l6, l7))
            buy_count = 0

        if sell_count == 9:
            last_sell9_idx = i
            if i >= 8:
                h8, h9 = high.iloc[i-1], high.iloc[i]
                h6, h7 = high.iloc[i-3], high.iloc[i-2]
                sell_perfected = (h8 >= max(h6, h7)) or (h9 >= max(h6, h7))
            sell_count = 0

    curr_buy = curr_sell = 0
    for i in range(max(4, n-8), n):
        if close.iloc[i] < close.iloc[i-4]:
            curr_buy += 1; curr_sell = 0
        elif close.iloc[i] > close.iloc[i-4]:
            curr_sell += 1; curr_buy = 0
        else:
            curr_buy = curr_sell = 0

    last_signal = "BUY9" if last_buy9_idx and (not last_sell9_idx or last_buy9_idx > last_sell9_idx) else \
                  "SELL9" if last_sell9_idx and (not last_buy9_idx or last_sell9_idx > last_buy9_idx) else "NONE"

    return {
        "last_signal": last_signal,
        "last_buy9_date": str(close.index[last_buy9_idx].date()) if last_buy9_idx else None,
        "last_sell9_date": str(close.index[last_sell9_idx].date()) if last_sell9_idx else None,
        "buy_perfected": bool(buy_perfected),
        "sell_perfected": bool(sell_perfected),
        "curr_buy_count": int(curr_buy),
        "curr_sell_count": int(curr_sell),
    }

## Compute indicators per ticker
In this section we loop through each series, compute all indicators, and assemble a tidy DataFrame with one row per symbol.

In [27]:
def compute_metrics(price_dict: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for t, df in price_dict.items():
        df = df.dropna()
        if df.empty or {"Open","High","Low","Close"}.difference(df.columns):
            continue

        close = df["Close"]

        rsi_ser = rsi(close)
        macd_line, macd_signal, macd_hist = macd(close)
        bb_mid, bb_up, bb_lo, bb_pctb, bb_bw = bollinger(close)
        hi52, lo52 = fifty_two_week_metrics(close)
        support, resistance = local_extrema_support_resistance(close, lookback=180)
        td = td_sequential_basic(df)

        last = df.iloc[-1]
        rows.append(dict(
            symbol=t,
            date=str(df.index[-1].date()),
            close=float(last["Close"]),
            change_1d=float((close.pct_change().iloc[-1] or 0) * 100),
            rsi=float(rsi_ser.iloc[-1]) if pd.notna(rsi_ser.iloc[-1]) else np.nan,
            macd=float(macd_line.iloc[-1]) if pd.notna(macd_line.iloc[-1]) else np.nan,
            macd_signal=float(macd_signal.iloc[-1]) if pd.notna(macd_signal.iloc[-1]) else np.nan,
            macd_hist=float(macd_hist.iloc[-1]) if pd.notna(macd_hist.iloc[-1]) else np.nan,
            bb_mid=float(bb_mid.iloc[-1]) if pd.notna(bb_mid.iloc[-1]) else np.nan,
            bb_upper=float(bb_up.iloc[-1]) if pd.notna(bb_up.iloc[-1]) else np.nan,
            bb_lower=float(bb_lo.iloc[-1]) if pd.notna(bb_lo.iloc[-1]) else np.nan,
            bb_pctb=float(bb_pctb.iloc[-1]) if pd.notna(bb_pctb.iloc[-1]) else np.nan,
            bb_bandwidth=float(bb_bw.iloc[-1]) if pd.notna(bb_bw.iloc[-1]) else np.nan,
            hi_52w=float(hi52), lo_52w=float(lo52),
            support=float(support) if pd.notna(support) else np.nan,
            resistance=float(resistance) if pd.notna(resistance) else np.nan,
            td_last_signal=td["last_signal"],
            td_last_buy9_date=td["last_buy9_date"],
            td_last_sell9_date=td["last_sell9_date"],
            td_buy_perfected=td["buy_perfected"],
            td_sell_perfected=td["sell_perfected"],
            td_curr_buy_count=td["curr_buy_count"],
            td_curr_sell_count=td["curr_sell_count"],
        ))
    return pd.DataFrame(rows)

In [28]:
metrics_df = compute_metrics(price_data).sort_values("symbol").reset_index(drop=True)
print(f"Computed metrics for {len(metrics_df)} tickers.")
metrics_df.head()

Computed metrics for 129 tickers.


,symbol,date,close,change_1d,rsi,macd,macd_signal,macd_hist,bb_mid,bb_upper,...,lo_52w,support,resistance,td_last_signal,td_last_buy9_date,td_last_sell9_date,td_buy_perfected,td_sell_perfected,td_curr_buy_count,td_curr_sell_count
0,AAL,2025-10-31,13.130000,2.738657,62.171628,0.266017,0.159943,0.106074,12.339000,13.594514,...,9.070000,13.000000,13.150000,BUY9,2025-09-29,2025-05-06,True,True,2,0
1,AAPL,2025-10-31,270.369995,-0.379513,80.532438,6.295855,5.302713,0.993142,258.509499,275.128934,...,172.419998,258.450012,271.399994,SELL9,2025-01-13,2025-10-28,True,True,0,8
2,ACHR,2025-10-31,11.220000,2.372265,34.833657,0.145710,0.319609,-0.173900,11.871500,13.508843,...,3.150000,11.140000,11.340000,SELL9,2025-08-01,2025-09-24,True,True,2,0
3,AFRM,2025-10-31,71.879997,4.325103,47.837652,-1.556848,-1.658506,0.101658,73.386500,78.742047,...,35.750000,71.139999,73.059998,SELL9,2025-03-04,2025-09-04,True,False,3,0
4,AI,2025-10-31,17.580000,3.229590,36.581714,-0.167791,-0.045722,-0.122069,18.338500,19.774791,...,15.460000,17.559999,17.950001,SELL9,2025-08-05,2025-09-22,False,True,3,0


Now that we have all the indicators for all the tickers in my watchlist, we are good to move on to derive the categorical field of a Buy/Hold/Sell scoring.

## Tagging & Local rule-based scoring
In this section we convert numeric indicators into categorical "tags" and then into a transparent numeric score. This score will later be adjusted by the LLM output to form the final Buy/Hold/Sell.

In [31]:
def build_tags(r) -> str:
    tags = []

    # RSI zones
    if pd.notna(r.rsi):
        if r.rsi >= 70: tags.append("RSI_overbought")
        elif r.rsi <= 30: tags.append("RSI_oversold")
        else: tags.append("RSI_neutral")

    # MACD momentum
    if pd.notna(r.macd_hist) and pd.notna(r.macd) and pd.notna(r.macd_signal):
        if r.macd_hist > 0 and r.macd >= r.macd_signal:
            tags.append("MACD_bullish")
        elif r.macd_hist < 0 and r.macd <= r.macd_signal:
            tags.append("MACD_bearish")
        else:
            tags.append("MACD_mixed")

    # Bollinger position & width
    if pd.notna(r.bb_pctb):
        if r.bb_pctb >= 0.9: tags.append("near_upper_band")
        elif r.bb_pctb <= 0.1: tags.append("near_lower_band")
        else: tags.append("inside_bands")
    if pd.notna(r.bb_bandwidth):
        if r.bb_bandwidth <= 0.05: tags.append("bollinger_squeeze")
        elif r.bb_bandwidth >= 0.25: tags.append("high_volatility_bandwidth")

    # 52w placement
    if pd.notna(r.hi_52w) and pd.notna(r.lo_52w) and pd.notna(r.close):
        rng = r.hi_52w - r.lo_52w
        pos = (r.close - r.lo_52w) / rng if rng > 0 else np.nan
        if pd.notna(pos):
            if pos >= 0.95: tags.append("near_52w_high")
            elif pos <= 0.05: tags.append("near_52w_low")

    # Support / Resistance proximity
    if pd.notna(r.support) and r.support > 0:
        if 0 < (r.close - r.support) / r.close <= 0.02:
            tags.append("on_support")
    if pd.notna(r.resistance) and r.resistance > 0:
        if 0 < (r.resistance - r.close) / r.close <= 0.02:
            tags.append("near_resistance")

    # TD Sequential
    if r.td_last_signal == "BUY9":  tags.append("TD_recent_buy9")
    if r.td_last_signal == "SELL9": tags.append("TD_recent_sell9")
    if r.td_buy_perfected:  tags.append("TD_buy_perfected")
    if r.td_sell_perfected: tags.append("TD_sell_perfected")
    if r.td_curr_buy_count >= 5:  tags.append(f"TD_buy_count_{r.td_curr_buy_count}")
    if r.td_curr_sell_count >= 5: tags.append(f"TD_sell_count_{r.td_curr_sell_count}")

    return " ".join(tags)

def local_score_from_tags(tags: str) -> int:
    s = 0
    if not isinstance(tags, str): return s
    parts = set(tags.split())

    # Momentum via MACD
    if "MACD_bullish" in parts: s += 2
    elif "MACD_bearish" in parts: s -= 2

    # RSI extremes
    if "RSI_oversold" in parts: s += 1
    elif "RSI_overbought" in parts: s -= 1

    # Bands location
    if "near_lower_band" in parts: s += 1
    elif "near_upper_band" in parts: s -= 1

    # 52w placement
    if "near_52w_low" in parts: s += 1
    elif "near_52w_high" in parts: s -= 1

    # Support/Resistance
    if "on_support" in parts: s += 1
    if "near_resistance" in parts: s -= 1

    # TD
    if "TD_recent_buy9" in parts: s += 2
    if "TD_recent_sell9" in parts: s -= 2
    if "TD_buy_perfected" in parts: s += 1
    if "TD_sell_perfected" in parts: s -= 1

    return int(s)

In [32]:
pd.set_option('display.max_columns', None)
metrics_df["tags"] = metrics_df.apply(build_tags, axis=1)
metrics_df["local_score"] = metrics_df["tags"].apply(local_score_from_tags)
metrics_df.head()

,symbol,date,close,change_1d,rsi,macd,macd_signal,macd_hist,bb_mid,bb_upper,bb_lower,bb_pctb,bb_bandwidth,hi_52w,lo_52w,support,resistance,td_last_signal,td_last_buy9_date,td_last_sell9_date,td_buy_perfected,td_sell_perfected,td_curr_buy_count,td_curr_sell_count,tags,local_score
0,AAL,2025-10-31,13.130000,2.738657,62.171628,0.266017,0.159943,0.106074,12.339000,13.594514,11.083486,0.815010,0.203503,18.660000,9.070000,13.000000,13.150000,BUY9,2025-09-29,2025-05-06,True,True,2,0,RSI_neutral MACD_bullish inside_bands on_suppo...,4
1,AAPL,2025-10-31,270.369995,-0.379513,80.532438,6.295855,5.302713,0.993142,258.509499,275.128934,241.890065,0.856826,0.128579,271.399994,172.419998,258.450012,271.399994,SELL9,2025-01-13,2025-10-28,True,True,0,8,RSI_overbought MACD_bullish inside_bands near_...,-3
2,ACHR,2025-10-31,11.220000,2.372265,34.833657,0.145710,0.319609,-0.173900,11.871500,13.508843,10.234157,0.301050,0.275844,13.640000,3.150000,11.140000,11.340000,SELL9,2025-08-01,2025-09-24,True,True,2,0,RSI_neutral MACD_bearish inside_bands high_vol...,-4
3,AFRM,2025-10-31,71.879997,4.325103,47.837652,-1.556848,-1.658506,0.101658,73.386500,78.742047,68.030953,0.359351,0.145955,92.180000,35.750000,71.139999,73.059998,SELL9,2025-03-04,2025-09-04,True,False,3,0,RSI_neutral MACD_bullish inside_bands on_suppo...,1
4,AI,2025-10-31,17.580000,3.229590,36.581714,-0.167791,-0.045722,-0.122069,18.338500,19.774791,16.902209,0.235952,0.156642,42.939999,15.460000,17.559999,17.950001,SELL9,2025-08-05,2025-09-22,False,True,3,0,RSI_neutral MACD_bearish inside_bands on_suppo...,-4


The results look good to go. Let's move on to the **GenAI** which is the focus of the course and thus project.

## GenAI prompt & call (OpenAI)
In this section we:
- Define a concise system prompt (tone + constraints).
- Build a compact JSON payload per row to keep tokens low.
- Ask the model to return a STRICT JSON with fields:
    * stance ∈ {bullish,bearish,neutral}, llm_score ∈ \[-2,2], summary (1–3 sentences), and optional key levels.
- Parse results and attach to the DataFrame.

In [35]:
if not api_key:
    raise RuntimeError("Please set OPENAI_API_KEY in your environment before running the GenAI step.")

client = OpenAI(api_key=api_key)

SYSTEM_PROMPT = """You are a disciplined technical analyst.
Write a concise, neutral read of the chart using the provided indicators.
- 1 to 3 sentences, no emojis or bullet points.
- No advice or targets; describe momentum, trend, and key nearby levels if present.
- Return STRICT JSON with keys: stance (bullish|bearish|neutral), llm_score (integer -2..2), summary (string <= 300 chars).
- Consider RSI/MACD/Bollinger/TD/52w/support/resistance holistically."""

def make_llm_input(row: pd.Series) -> str:
    payload = {
        "symbol": row.symbol,
        "date": row.date,
        "close": round(float(row.close), 4) if pd.notna(row.close) else None,
        "change_1d_pct": round(float(row.change_1d), 2) if pd.notna(row.change_1d) else None,
        "rsi": round(float(row.rsi), 2) if pd.notna(row.rsi) else None,
        "macd": round(float(row.macd), 6) if pd.notna(row.macd) else None,
        "macd_signal": round(float(row.macd_signal), 6) if pd.notna(row.macd_signal) else None,
        "macd_hist": round(float(row.macd_hist), 6) if pd.notna(row.macd_hist) else None,
        "bb_mid": round(float(row.bb_mid), 4) if pd.notna(row.bb_mid) else None,
        "bb_upper": round(float(row.bb_upper), 4) if pd.notna(row.bb_upper) else None,
        "bb_lower": round(float(row.bb_lower), 4) if pd.notna(row.bb_lower) else None,
        "bb_pctb": round(float(row.bb_pctb), 4) if pd.notna(row.bb_pctb) else None,
        "bb_bandwidth": round(float(row.bb_bandwidth), 4) if pd.notna(row.bb_bandwidth) else None,
        "hi_52w": round(float(row.hi_52w), 4) if pd.notna(row.hi_52w) else None,
        "lo_52w": round(float(row.lo_52w), 4) if pd.notna(row.lo_52w) else None,
        "support": round(float(row.support), 4) if pd.notna(row.support) else None,
        "resistance": round(float(row.resistance), 4) if pd.notna(row.resistance) else None,
        "td_last_signal": row.td_last_signal,
        "td_curr_buy_count": int(row.td_curr_buy_count),
        "td_curr_sell_count": int(row.td_curr_sell_count),
        "tags": row.tags.split() if isinstance(row.tags, str) else [],
        # Guidance for scoring: model maps holistic read into -2..2
        "scoring_guidance": {
            "bullish":  [1, 2],
            "neutral":  [0],
            "bearish":  [-1, -2],
            "notes": "Use -2/+2 only when evidence is strong across indicators."
        }
    }
    return json.dumps(payload, ensure_ascii=False)

def call_llm_for_row(row: pd.Series) -> dict:
    try:
        user_prompt = (
            "Analyze the following JSON of daily technicals and return STRICT JSON as instructed.\n"
            f"JSON:\n{make_llm_input(row)}"
        )
        resp = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {"role":"system","content":SYSTEM_PROMPT},
                {"role":"user","content":user_prompt},
            ],
            temperature=LLM_TEMPERATURE,
            max_tokens=180,
        )
        text = resp.choices[0].message.content.strip()
        # Try to parse strict JSON
        parsed = json.loads(text)
        # Validate minimal schema
        stance = parsed.get("stance","neutral")
        llm_score = int(parsed.get("llm_score", 0))
        summary = parsed.get("summary","")
        if stance not in {"bullish","bearish","neutral"}:
            stance = "neutral"
        llm_score = max(-2, min(2, llm_score))
        return {"llm_stance": stance, "llm_score": llm_score, "llm_summary": summary}
    except Exception as e:
        return {"llm_stance":"neutral","llm_score":0,"llm_summary":f"[LLM error] {e}"}

# Only summarize "interesting" tickers to cut cost as an option:
def interesting_row(r) -> bool:
    if SUMMARIZE_ALL_TICKERS:
        return True
    tags = str(r.tags)
    return any(k in tags for k in ["bollinger_squeeze","TD_recent_buy9","near_resistance","near_52w_high","near_52w_low"])

llm_results = []
for _, row in metrics_df.iterrows():
    if interesting_row(row):
        res = call_llm_for_row(row)
        time.sleep(SLEEP_BETWEEN_CALLS_SEC)
    else:
        res = {"llm_stance":"neutral","llm_score":0,"llm_summary":"(LLM skipped to save cost)"}
    llm_results.append(res)

In [36]:
metrics_df = pd.concat([metrics_df, pd.DataFrame(llm_results)], axis=1)
metrics_df.head()

,symbol,date,close,change_1d,rsi,macd,macd_signal,macd_hist,bb_mid,bb_upper,bb_lower,bb_pctb,bb_bandwidth,hi_52w,lo_52w,support,resistance,td_last_signal,td_last_buy9_date,td_last_sell9_date,td_buy_perfected,td_sell_perfected,td_curr_buy_count,td_curr_sell_count,tags,local_score,llm_stance,llm_score,llm_summary
0,AAL,2025-10-31,13.130000,2.738657,62.171628,0.266017,0.159943,0.106074,12.339000,13.594514,11.083486,0.815010,0.203503,18.660000,9.070000,13.000000,13.150000,BUY9,2025-09-29,2025-05-06,True,True,2,0,RSI_neutral MACD_bullish inside_bands on_suppo...,4,bullish,1,The stock shows bullish momentum with an RSI o...
1,AAPL,2025-10-31,270.369995,-0.379513,80.532438,6.295855,5.302713,0.993142,258.509499,275.128934,241.890065,0.856826,0.128579,271.399994,172.419998,258.450012,271.399994,SELL9,2025-01-13,2025-10-28,True,True,0,8,RSI_overbought MACD_bullish inside_bands near_...,-3,bearish,-1,AAPL shows overbought conditions with an RSI o...
2,ACHR,2025-10-31,11.220000,2.372265,34.833657,0.145710,0.319609,-0.173900,11.871500,13.508843,10.234157,0.301050,0.275844,13.640000,3.150000,11.140000,11.340000,SELL9,2025-08-01,2025-09-24,True,True,2,0,RSI_neutral MACD_bearish inside_bands high_vol...,-4,bearish,-1,The momentum is bearish with the RSI at 34.83 ...
3,AFRM,2025-10-31,71.879997,4.325103,47.837652,-1.556848,-1.658506,0.101658,73.386500,78.742047,68.030953,0.359351,0.145955,92.180000,35.750000,71.139999,73.059998,SELL9,2025-03-04,2025-09-04,True,False,3,0,RSI_neutral MACD_bullish inside_bands on_suppo...,1,neutral,0,The stock is currently at support near 71.14 a...
4,AI,2025-10-31,17.580000,3.229590,36.581714,-0.167791,-0.045722,-0.122069,18.338500,19.774791,16.902209,0.235952,0.156642,42.939999,15.460000,17.559999,17.950001,SELL9,2025-08-05,2025-09-22,False,True,3,0,RSI_neutral MACD_bearish inside_bands on_suppo...,-4,bearish,-1,The stock is showing bearish momentum with an ...


Now that we get both the local and LLM indicators and scores, let's combine them for a final decision.

## Final decision blending (Local + GenAI)
In this section we combine the local_score with the LLM score. This keeps the process transparent while letting GenAI influence the call.

In [39]:
def final_score(local_score: int, llm_score: int) -> int:
    # Blend: weight local >= LLM to stay conservative (e.g., local + llm*1)
    return int(local_score + llm_score)

def decision_from_score(score: int) -> str:
    if score >= SCORE_THRESH_BUY:  return "BUY"
    if score <= SCORE_THRESH_SELL: return "SELL"
    return "HOLD"

metrics_df["final_score"] = metrics_df.apply(lambda r: final_score(int(r.local_score), int(r.llm_score)), axis=1)
metrics_df["decision"] = metrics_df["final_score"].apply(decision_from_score)

# Keep nice display columns
display_df = metrics_df.copy()
for col in ["close","change_1d","rsi","macd","macd_signal","macd_hist",
            "bb_mid","bb_upper","bb_lower","bb_pctb","bb_bandwidth",
            "hi_52w","lo_52w","support","resistance"]:
    if col in display_df:
        display_df[col] = pd.to_numeric(display_df[col], errors="coerce").astype(float).round(4)

# Merge back original symbols for reference
display_df = display_df.merge(
    watch_df[["symbol","symbol_original"]],
    on="symbol",
    how="left"
)

display_df = display_df[[
    "symbol_original","symbol","date","close","change_1d",
    "rsi","macd","macd_signal","macd_hist",
    "bb_pctb","bb_bandwidth","support","resistance",
    "hi_52w","lo_52w","td_last_signal","td_curr_buy_count","td_curr_sell_count",
    "tags","local_score","llm_stance","llm_score","llm_summary",
    "final_score","decision"
]].sort_values(["decision","final_score","symbol"], ascending=[True, False, True]).reset_index(drop=True)

In [40]:
display_df.to_csv(CSV_OUT, index=False)
print(f"Saved metrics CSV ⇒ {CSV_OUT}")
display_df.head(8)

Saved metrics CSV ⇒ ./watchlist_metrics_20251102.csv


,symbol_original,symbol,date,close,change_1d,rsi,macd,macd_signal,macd_hist,bb_pctb,bb_bandwidth,support,resistance,hi_52w,lo_52w,td_last_signal,td_curr_buy_count,td_curr_sell_count,tags,local_score,llm_stance,llm_score,llm_summary,final_score,decision
0,AAL.O,AAL,2025-10-31,13.13,2.7387,62.1716,0.2660,0.1599,0.1061,0.8150,0.2035,13.00,13.15,18.66,9.07,BUY9,2,0,RSI_neutral MACD_bullish inside_bands on_suppo...,4,bullish,1,The stock shows bullish momentum with an RSI o...,5,BUY
1,CRM,CRM,2025-10-31,260.41,1.4650,58.6989,3.3955,2.4016,0.9939,0.8612,0.1213,259.50,262.38,367.87,231.66,BUY9,0,2,RSI_neutral MACD_bullish inside_bands on_suppo...,4,bullish,1,The momentum is bullish with the MACD above it...,5,BUY
2,KLAR.K,KLAR,2025-10-31,37.57,2.6222,45.1094,-0.9346,-1.0756,0.1410,0.3978,0.2151,36.65,39.30,45.48,35.28,BUY9,2,0,RSI_neutral MACD_bullish inside_bands TD_recen...,5,neutral,0,The momentum is neutral with an RSI of 45.11 a...,5,BUY
3,ALAB.O,ALAB,2025-10-31,186.68,10.1032,43.2567,-7.5649,-9.3491,1.7842,0.5725,0.5303,185.85,189.15,251.88,52.94,BUY9,0,7,RSI_neutral MACD_bullish inside_bands high_vol...,4,neutral,0,"The stock is currently at 186.68, near support...",4,BUY
4,AVGO.O,AVGO,2025-10-31,369.63,-1.8169,56.5654,9.9241,7.3857,2.5384,0.7984,0.1676,359.63,385.98,385.98,146.29,BUY9,0,6,RSI_neutral MACD_bullish inside_bands TD_recen...,4,neutral,0,AVGO shows a neutral momentum with an RSI of 5...,4,BUY
5,DIS,DIS,2025-10-31,112.62,0.6974,57.6052,-0.4908,-0.6815,0.1907,0.6924,0.0433,112.53,112.75,124.01,81.72,BUY9,0,2,RSI_neutral MACD_bullish inside_bands bollinge...,4,neutral,0,The stock is currently on support at 112.53 an...,4,BUY
6,FTNT.O,FTNT,2025-10-31,86.43,2.6485,61.5784,0.4267,0.3798,0.0469,0.8728,0.0528,85.79,86.46,114.57,74.39,BUY9,0,1,RSI_neutral MACD_bullish inside_bands on_suppo...,4,neutral,0,FTNT shows a neutral momentum with an RSI of 6...,4,BUY
7,GRAB.O,GRAB,2025-10-31,6.01,0.5017,53.0303,0.0256,0.0151,0.0105,0.5793,0.1573,5.98,6.05,6.45,3.48,BUY9,1,0,RSI_neutral MACD_bullish inside_bands on_suppo...,4,neutral,0,The momentum is stable with an RSI of 53.03 an...,4,BUY


Data looks good. Next, we will focus on full fine tuning.

## Fine Tuning
For this section, I’ll fine-tune a small “LLM stance head” that replaces the hand-crafted prompt from the previous sections as it's required in the project milestone. The model will learn to map my engineered inputs (normalized symbol + local metrics I computed from Yahoo Finance) --> llm_stance, llm_score, llm_summary. I'll keep my existing policy to combine local_score + llm_score for final score/decision exactly as before. Now let's build the training JSONL for fine-tuning this “stance head.”

### Building the training JOSNL for fine tuning
For this subsection, I will build one fine-tuning dataset where the model learns to map:
- Inputs: symbol (normalized) + local metrics/features (including tags)
- Targets: llm_stance, llm_score, llm_summary (from Milestone 2, I'll exclude llm_* target columns from features to avoid leakage.)
- Output file: ./m3_llmhead_all.jsonl

In [85]:
from pathlib import Path
import json, re
import pandas as pd

# === Milestone 2 CSV ===
CSV_PATH = Path("./watchlist_metrics_20251102.csv")  # update if needed

# === REQUIRED column names (targets we want to learn) ===
symbol_col       = "symbol"        # normalized ticker (e.g., "AAL")
date_col         = None            # optional, e.g., "as_of" or "date"
llm_stance_col   = "llm_stance"    # the prior LLM stance labels
llm_score_col    = "llm_score"     # the prior LLM numeric score
llm_summary_col  = "llm_summary"   # the prior LLM summary text

# === Optional (if these exist, we'll explicitly exclude them from features) ===
final_score_cols    = ["final_score", "combined_final_score"] # I used final_score in the prior milestone
final_decision_cols = ["final_decision", "decision", "action", "recommendation"] # I used the decision in the prior milestone

# === Output ===
OUT_JSONL = Path("./m3_llmhead_all.jsonl")

In [87]:
def normalize_stance(x: str):
    """Canonicalize stance. I can also keep the original mapping if I prefer later."""
    if not isinstance(x, str):
        return None
    t = x.strip().lower()
    if t in {"bull", "bullish"}:   return "BULLISH"
    if t in {"bear", "bearish"}:   return "BEARISH"
    if t in {"neutral", "sideways"}: return "NEUTRAL"
    return x.strip().upper()

def compact_float(x, sig=6):
    if pd.isna(x): return None
    try:
        return float(f"{float(x):.{sig}g}")
    except Exception:
        return None

def safe_str(x):
    return None if pd.isna(x) else str(x)

def shorten(text: str, max_chars=1000):
    if not isinstance(text, str): return None
    text = text.strip()
    return text if len(text) <= max_chars else text[:max_chars] + "…"

In [89]:
# Load data
assert CSV_PATH.exists(), f"CSV not found: {CSV_PATH}"
df = pd.read_csv(CSV_PATH)

# Sanity: required columns exist
required = [symbol_col, llm_stance_col, llm_score_col, llm_summary_col]
missing = [c for c in required if c not in df.columns]
assert not missing, f"Missing required columns in CSV: {missing}"

# Assemble exclusion set (targets + bookkeeping)
exclude_cols = {symbol_col, llm_stance_col, llm_score_col, llm_summary_col}
if date_col and date_col in df.columns:
    exclude_cols.add(date_col)

for c in final_score_cols + final_decision_cols:
    if c in df.columns:
        exclude_cols.add(c)

# Choose feature columns:
# - include numeric engineered metrics
# - include compact categorical/tag cols (e.g., tagging strings)
feature_cols = []
for c in df.columns:
    if c in exclude_cols:
        continue
    if pd.api.types.is_numeric_dtype(df[c]):
        feature_cols.append(c)
    else:
        # allow “tag/label/flag/signal/state” style engineered strings
        if re.search(r"(tag|label|flag|signal|state|pattern|setup|td_last_signal)", c, flags=re.I):
            feature_cols.append(c)

# (Optional) deterministic order for features to stabilize training inputs
feature_cols = sorted(feature_cols)

# Build examples
examples = []
for _, row in df.iterrows():
    # Inputs
    features = {"symbol": safe_str(row[symbol_col])}
    if date_col and date_col in df.columns:
        features["as_of"] = safe_str(row[date_col])

    for c in feature_cols:
        if pd.api.types.is_numeric_dtype(df[c]):
            num = compact_float(row[c])
            if num is not None:
                features[c] = num
        else:
            s = safe_str(row[c])
            if s:
                features[c] = s

    # Targets (from the Milestone 2 LLM pass)
    stance  = normalize_stance(row[llm_stance_col])
    score   = compact_float(row[llm_score_col])
    summary = shorten(safe_str(row[llm_summary_col]), max_chars=1200)

    if not stance or score is None or not summary:
        continue  # skip incomplete targets

    assistant_text = f"stance: {stance}\nscore: {score}\nsummary: {summary}"

    system_text = (
        "You are a professional stock analyst and an equity stance assistant. "
        "Given engineered metrics for a ticker, output ONLY:\n"
        "stance: BULLISH|NEUTRAL|BEARISH\n"
        "score: <float>\n"
        "summary: <one or two short sentences>\n"
        "Do not fetch external data. Be concise and deterministic. score should be integer (-2..2)."
    )

    user_text = (
        "Derive stance, score, and a concise summary from the following engineered snapshot:\n"
        + json.dumps(features, ensure_ascii=False)
    )

    examples.append({
        "messages": [
            {"role": "system", "content": system_text},
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": assistant_text},
        ]
    })

assert len(examples) > 0, "No supervised examples created. Check that llm_* columns are populated."

# Write single JSONL (no split)
with OUT_JSONL.open("w", encoding="utf-8") as f:
    for ex in examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print(f"Wrote {len(examples)} examples to {OUT_JSONL.resolve()}")

Wrote 129 examples to /Users/ZichengXu/Desktop/DSC 670/Term Project/m3_llmhead_all.jsonl


In [95]:
# View a few examples for sanity-check
import itertools
for ex in itertools.islice(examples, 2):
    print(json.dumps(ex, ensure_ascii=False)[:1500] + "\n---\n")

{"messages": [{"role": "system", "content": "You are a professional stock analyst and an equity stance assistant. Given engineered metrics for a ticker, output ONLY:\nstance: BULLISH|NEUTRAL|BEARISH\nscore: <float>\nsummary: <one or two short sentences>\nDo not fetch external data. Be concise and deterministic. score should be integer (-2..2)."}, {"role": "user", "content": "Derive stance, score, and a concise summary from the following engineered snapshot:\n{\"symbol\": \"AAL\", \"bb_bandwidth\": 0.2035, \"bb_pctb\": 0.815, \"change_1d\": 2.7387, \"close\": 13.13, \"hi_52w\": 18.66, \"lo_52w\": 9.07, \"local_score\": 4.0, \"macd\": 0.266, \"macd_hist\": 0.1061, \"macd_signal\": 0.1599, \"resistance\": 13.15, \"rsi\": 62.1716, \"support\": 13.0, \"tags\": \"RSI_neutral MACD_bullish inside_bands on_support near_resistance TD_recent_buy9 TD_buy_perfected TD_sell_perfected\", \"td_curr_buy_count\": 2.0, \"td_curr_sell_count\": 0.0, \"td_last_signal\": \"BUY9\"}"}, {"role": "assistant", "c

The preview shows the **stance-head dataset** is well-formed: 129 chat-style examples where the **user** provides engineered features (normalized `symbol`, Bollinger stats, 1-day change, close, 52-week high/low, `local_score`, full MACD triplet, RSI, **support/resistance**, TD counts/signals, and a compact tag string), and the **assistant** returns a tight schema—`stance` (canonicalized to **BULLISH/NEUTRAL/BEARISH**), `score` constrained to the **−2..2** integer scale (examples use **1.0**, which is fine but I may choose to output plain `1` to reinforce “integer”), and a **1–2 sentence summary** grounded in the same inputs (e.g., “MACD above signal,” “RSI ~60,” “near resistance,” “inside bands”). Both samples (AAL, CRM) are **BULLISH** with consistent rationale patterns (MACD strength + RSI neutral-to-positive + proximity to support/resistance), which matches how my Milestone-2 prompt behaved—good for reproducibility via fine-tuning. Overall, the schema is deterministic, avoids external data, and the features reflect precisely the signal set I want the model to learn. 

In [115]:
# Milestone 3 — Token & Cost Analysis (stance-head dataset)
from collections import defaultdict
from pathlib import Path
import json
import numpy as np

import tiktoken
encoding = tiktoken.get_encoding("cl100k_base")

# === Path to the single JSONL I built in previous Step ===
DATA_FILE = Path("./m3_llmhead_all.jsonl")  
assert DATA_FILE.exists(), f"Training file not found: {DATA_FILE}"

# === Context window cap for per-example billable tokens (book-aligned) ===
MAX_TOKENS = 4096

# === Epoch heuristic ===
TARGET_EPOCHS = 3
MIN_TARGET_EXAMPLES = 100
MAX_TARGET_EXAMPLES = 25000
MIN_DEFAULT_EPOCHS = 1
MAX_DEFAULT_EPOCHS = 25

# === cost model (I will fill with the actual FT training rate if I want $ estimate) ===
# Many users maintain a small config table per base model; here we keep it simple & explicit.
TRAIN_PRICE_PER_1K_TOKENS = None  # e.g., 0.008 for $0.008 / 1K tokens (set to a float to enable cost calc)
BASE_MODEL = "gpt-4o-mini-2024-07-18"        # just for labeling in the report

In [117]:
def basic_checks(data_file: Path) -> bool:
    """Quick sanity checks, preview first example."""
    try:
        with data_file.open("r", encoding="utf-8") as f:
            dataset = [json.loads(line) for line in f]

        print(f"Basic checks for file {data_file}:")
        print("Count of examples in training dataset:", len(dataset))
        print("First example:")
        for message in dataset[0]["messages"]:
            print(message)
        return True
    except FileNotFoundError as e:
        print(f"File not found: {e}")
        return False
    except json.JSONDecodeError as e:
        print(f"JSON decoding error: {e}")
        return False
    except Exception as e:
        print(f"Unexpected error: {e}")
        return False

def format_checks(dataset, filename) -> bool:
    """Non-exhaustive schema checks for chat fine-tuning format."""
    format_errors = defaultdict(int)

    for ex in dataset:
        if not isinstance(ex, dict):
            format_errors["data_type"] += 1
            continue

        messages = ex.get("messages")
        if not messages or not isinstance(messages, list):
            format_errors["missing_messages_list"] += 1
            continue

        for message in messages:
            if "role" not in message or "content" not in message:
                format_errors["message_missing_key"] += 1

            # Only allow standard chat keys
            if any(k not in ("role", "content", "name", "function_call") for k in message):
                format_errors["message_unrecognized_key"] += 1

            if message.get("role") not in ("system", "user", "assistant", "function"):
                format_errors["unrecognized_role"] += 1

            content = message.get("content")
            function_call = message.get("function_call")
            if (not content and not function_call) or not isinstance(content, str):
                format_errors["missing_content"] += 1

        if not any(m.get("role") == "assistant" for m in messages):
            format_errors["example_missing_assistant_message"] += 1

    if format_errors:
        print(f"Formatting errors found in file {filename}:")
        for k, v in format_errors.items():
            print(f"  {k}: {v}")
        return False

    print(f"No formatting errors found in file {filename}")
    return True

def num_tokens_from_messages(messages, tokens_per_message=3, tokens_per_name=1):
    """Rough token estimate matching common chat-count heuristics (as in my Week 6 code)."""
    num_tokens = 0
    for message in messages:
        num_tokens += tokens_per_message
        for key, value in message.items():
            if isinstance(value, str):
                num_tokens += len(encoding.encode(value))
            if key == "name":
                num_tokens += tokens_per_name
    num_tokens += 3  # reply primer
    return num_tokens

def num_assistant_tokens_from_messages(messages):
    """Assistant-only token count (used for training billing in my Week 6 heuristic)."""
    num_tokens = 0
    for message in messages:
        if message.get("role") == "assistant":
            num_tokens += len(encoding.encode(message.get("content", "")))
    return num_tokens

def print_distribution(values, name):
    values = list(values)
    print(f"\n#### Distribution of {name}:")
    print(f"min / max: {min(values)} / {max(values)}")
    print(f"mean / median: {np.mean(values):.1f} / {np.median(values):.1f}")
    print(f"p10 / p90: {np.quantile(values, 0.10):.1f} / {np.quantile(values, 0.90):.1f}")

def estimate_epochs(n_examples: int) -> int:
    """Week-6 style epoch heuristic."""
    n_epochs = TARGET_EPOCHS
    if n_examples * TARGET_EPOCHS < MIN_TARGET_EXAMPLES:
        n_epochs = min(MAX_DEFAULT_EPOCHS, max(1, MIN_TARGET_EXAMPLES // max(1, n_examples)))
    elif n_examples * TARGET_EPOCHS > MAX_TARGET_EXAMPLES:
        n_epochs = max(MIN_DEFAULT_EPOCHS, MAX_TARGET_EXAMPLES // max(1, n_examples))
    return int(n_epochs)

def estimate_training_tokens(dataset, assistant_tokens_per_ex):
    """
    Book-aligned estimate of billable training tokens:
    sum(min(MAX_TOKENS, assistant_tokens_per_example)) * n_epochs
    """
    n_epochs = estimate_epochs(len(dataset))
    billed_per_dataset = sum(min(MAX_TOKENS, t) for t in assistant_tokens_per_ex)
    total_billable = n_epochs * billed_per_dataset
    return n_epochs, billed_per_dataset, total_billable

In [119]:
# Load & validate
assert basic_checks(DATA_FILE), "Basic checks failed"

with DATA_FILE.open("r", encoding="utf-8") as f:
    dataset = [json.loads(line) for line in f]

assert format_checks(dataset, str(DATA_FILE)), "Format checks failed"

# Token counts
total_tokens = []
assistant_tokens = []

for ex in dataset:
    msgs = ex["messages"]
    total_tokens.append(num_tokens_from_messages(msgs))
    assistant_tokens.append(num_assistant_tokens_from_messages(msgs))

# Summary stats
print_distribution(total_tokens, "total tokens per example")
print_distribution(assistant_tokens, "assistant tokens per example")

# Training-token estimate (book-aligned)
n_epochs, billed_per_dataset, total_billable = estimate_training_tokens(dataset, assistant_tokens)

print("\n===== Training Token Estimate (Book-Aligned) =====")
print(f"Examples: {len(dataset)}")
print(f"Epochs (heuristic): {n_epochs}")
print(f"Per-dataset billable tokens (capped per example at {MAX_TOKENS}): ~{billed_per_dataset:,}")
print(f"Total billable tokens (epochs × per-dataset): ~{total_billable:,}")

# Optional: $ estimate (only if I set TRAIN_PRICE_PER_1K_TOKENS above)
if TRAIN_PRICE_PER_1K_TOKENS is not None:
    cost = (total_billable / 1000.0) * TRAIN_PRICE_PER_1K_TOKENS
    print(f"Approx training cost @ ${TRAIN_PRICE_PER_1K_TOKENS:.4f} / 1K tokens: ${cost:,.2f}")
else:
    print("\n(Set TRAIN_PRICE_PER_1K_TOKENS to a float to print a $ estimate.)")

Basic checks for file m3_llmhead_all.jsonl:
Count of examples in training dataset: 129
First example:
{'role': 'system', 'content': 'You are a professional stock analyst and an equity stance assistant. Given engineered metrics for a ticker, output ONLY:\nstance: BULLISH|NEUTRAL|BEARISH\nscore: <float>\nsummary: <one or two short sentences>\nDo not fetch external data. Be concise and deterministic. score should be integer (-2..2).'}
{'role': 'user', 'content': 'Derive stance, score, and a concise summary from the following engineered snapshot:\n{"symbol": "AAL", "bb_bandwidth": 0.2035, "bb_pctb": 0.815, "change_1d": 2.7387, "close": 13.13, "hi_52w": 18.66, "lo_52w": 9.07, "local_score": 4.0, "macd": 0.266, "macd_hist": 0.1061, "macd_signal": 0.1599, "resistance": 13.15, "rsi": 62.1716, "support": 13.0, "tags": "RSI_neutral MACD_bullish inside_bands on_support near_resistance TD_recent_buy9 TD_buy_perfected TD_sell_perfected", "td_curr_buy_count": 2.0, "td_curr_sell_count": 0.0, "td_last

The validated dataset looks clean and consistent: 129 chat examples with no formatting issues, compact inputs, and concise targets. Token stats are tight—total tokens per example cluster around ~379 (p10–p90 ≈ 369–387), and assistant tokens average ~74 (min/max 64–89), which tells me my summaries are brief and within the intended style. Using the Week-6 heuristic at 3 epochs, I estimate ~9,572 billable assistant tokens per pass and ~28,716 total across training—small enough that costs should be negligible once I plug in the per-1K rate. Overall, this stance-head dataset is well-bounded (far below the 4K cap), uniform, and ready to fine-tune with predictable training footprint.

### Upload & Fine-Tune (with event tracking)
For this sub section, I will upload my dataset and kick off a full fine-tune for the “stance head,” and log training events/metrics (per the course’s Chapter 9 style):
- Upload m3_llmhead_all.jsonl
- Create a fine-tuning job on a chat-fine-tuneable model
- Stream training events (loss, steps, checkpoints) and persist them
- Store job_id and the fine-tuned model name for later inference

In [99]:
# 1) Setup OpenAI client
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY missing. Set it in the environment or .env file."

# OpenAI Python SDK ≥ v1.0 style
from openai import OpenAI
client = OpenAI()

In [107]:
# 2) Config — update if needed
DATASET_PATH = Path("./m3_llmhead_all.jsonl")
assert DATASET_PATH.exists(), f"Training file not found: {DATASET_PATH}"

# I'll try "gpt-4o-mini" first. Will switch if my account doesn’t allow it.
BASE_MODEL = "gpt-4o-mini-2024-07-18"

# A short suffix (Should appear in my model name on completion)
MODEL_SUFFIX = "stance_head_v1"

ARTIFACTS_DIR = Path("./m3_artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)
JOB_META_PATH = ARTIFACTS_DIR / "ft_job_meta.json"
EVENTS_LOG_PATH = ARTIFACTS_DIR / "ft_events_log.jsonl"

In [103]:
# 3) Upload the JSONL to OpenAI files
# Purpose: provides a file_id to reference in the fine-tuning job

with open(DATASET_PATH, "rb") as f:
    up = client.files.create(file=f, purpose="fine-tune")

file_id = up.id
print("Uploaded:", file_id)

Uploaded: file-WTsDBUxdiYgTBFNyrbPG9w


In [109]:
# 4) Create the fine-tune job

job = client.fine_tuning.jobs.create(
    training_file=file_id,
    model=BASE_MODEL,
    suffix=MODEL_SUFFIX,
)

job_id = job.id
print("Created job:", job_id)

# Persist metadata early
with open(JOB_META_PATH, "w") as f:
    json.dump({"job_id": job_id, "training_file_id": file_id, "base_model": BASE_MODEL}, f, indent=2)
print("Wrote:", JOB_META_PATH)

Created job: ftjob-gCYjvn0r77dR9FHAgXkd64li
Wrote: m3_artifacts/ft_job_meta.json


In [113]:
# 5) Poll job status and append events to a JSONL log

import json, time
from datetime import datetime

def append_event(rec):
    with open(EVENTS_LOG_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def list_events_compat(client, job_id, limit=50, after=None):
    """
    Tries the current API first:
      client.fine_tuning.jobs.list_events(fine_tuning_job_id=..., limit=..., after=...)
    Falls back to older/alternate signatures if needed.
    """
    try:
        return client.fine_tuning.jobs.list_events(
            fine_tuning_job_id=job_id,
            limit=limit,
            after=after
        )
    except TypeError:
        # Older variants sometimes used 'id' or omitted 'after'
        try:
            return client.fine_tuning.jobs.list_events(
                id=job_id,
                limit=limit
            )
        except TypeError:
            # Very old fallback: try without keywords
            return client.fine_tuning.jobs.list_events(job_id, limit)

print("Polling events for job:", job_id)
seen_event_ids = set()
terminal_states = {"succeeded", "failed", "cancelled"}
after_cursor = None  # advance as we read new pages

while True:
    job = client.fine_tuning.jobs.retrieve(job_id)
    status = job.status

    # Fetch recent events (paged); de-dup by event id
    ev_page = list_events_compat(client, job_id, limit=50, after=after_cursor)
    new_any = False

    # The response is a paginated object with .data and (sometimes) .has_more / .last_id
    events = getattr(ev_page, "data", []) or []
    for ev in events:
        if ev.id in seen_event_ids:
            continue
        seen_event_ids.add(ev.id)
        rec = {
            "ts_iso": datetime.utcnow().isoformat() + "Z",
            "job_id": job_id,
            "type": getattr(ev, "type", None),
            "level": getattr(ev, "level", None),
            "message": getattr(ev, "message", None),
            "data": getattr(ev, "data", None),
            "status": status,
        }
        append_event(rec)
        msg = getattr(ev, "message", "") or getattr(ev, "type", "")
        print(f"[{status}] {msg}")
        new_any = True

    # Advance cursor if available
    after_cursor = getattr(ev_page, "last_id", after_cursor)

    if status in terminal_states:
        print("Terminal status:", status)
        break

    if not new_any:
        # keep me informed even if no new events this cycle
        print(f"[{status}] (no new events)")
    time.sleep(5)

Polling events for job: ftjob-gCYjvn0r77dR9FHAgXkd64li


/var/folders/39/wtkk07qn4f72st8rjrrz8brm0000gp/T/ipykernel_86102/921210604.py:53: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ts_iso": datetime.utcnow().isoformat() + "Z",


[running] Step 71/387: training loss=0.22
[running] Step 70/387: training loss=0.27
[running] Step 69/387: training loss=0.50
[running] Step 68/387: training loss=0.26
[running] Step 67/387: training loss=0.53
[running] Step 66/387: training loss=0.33
[running] Step 65/387: training loss=0.33
[running] Step 64/387: training loss=0.28
[running] Step 63/387: training loss=0.38
[running] Step 62/387: training loss=0.43
[running] Step 61/387: training loss=0.36
[running] Step 60/387: training loss=0.48
[running] Step 59/387: training loss=0.21
[running] Step 58/387: training loss=0.11
[running] Step 57/387: training loss=0.44
[running] Step 56/387: training loss=0.64
[running] Step 55/387: training loss=0.51
[running] Step 54/387: training loss=0.69
[running] Step 53/387: training loss=0.33
[running] Step 52/387: training loss=0.23
[running] Step 51/387: training loss=0.40
[running] Step 50/387: training loss=0.23
[running] Step 49/387: training loss=0.46
[running] Step 48/387: training lo

I see a healthy end-to-end training run. The event stream is slightly out of order (normal with paged delivery), but the signal is clear: early losses bounce as high as ~0.80 (e.g., step 31) and trend steadily lower into the 0.03–0.10 range by the 330–360s, which indicates good convergence on this small stance-head task. Two checkpoints were created (steps **129** and **258**), then policy/moderation evaluations completed and the job **succeeded**, so the model is enabled for sampling. Occasional upward blips (e.g., 0.33–0.60 around steps 150–170 or the odd 0.30 later) are typical mini-batch variance, not a red flag given the dominant downward trend and low terminal losses. The **UTC deprecation warning** is just from `datetime.utcnow()` in my logger; I can switch to `datetime.now(datetime.UTC)` to silence it. Overall, the training behaved as expected—loss decayed smoothly, checkpoints saved, compliance checks passed—so I’m ready to capture the fine-tuned model name and move to validation against the baseline.

In [128]:
# 6) Once the job is terminal, capture the resulting fine-tuned model name

job = client.fine_tuning.jobs.retrieve(job_id)
ft_model = getattr(job, "fine_tuned_model", None)
print("fine_tuned_model:", ft_model)

# Update job meta file
meta = {}
if JOB_META_PATH.exists():
    meta = json.loads(JOB_META_PATH.read_text())
meta.update({"status": job.status, "fine_tuned_model": ft_model})
with open(JOB_META_PATH, "w") as f:
    json.dump(meta, f, indent=2)
print("Updated:", JOB_META_PATH)

fine_tuned_model: ft:gpt-4o-mini-2024-07-18:personal:stance-head-v1:CXXVxeHu
Updated: m3_artifacts/ft_job_meta.json


### Inference on an example + persistence for Streamlit
For this sub section I will use the fine-tuned stance head to score a row from the original CSV, compare to the original llm_stance / llm_score / llm_summary, and then persist everything so next milestone I can import it directly in Streamlit without rerunning training:
- Load the fine-tuned model name from m3_artifacts/ft_job_meta.json.
- Rebuild the exact feature snapshot from a chosen row in watchlist_metrics_20251102.csv.
- Call the fine-tuned stance head and parses stance/score/summary.
- Compare against the three LLM columns in the CSV (llm_stance, llm_score, llm_summary) with light metrics.
- Save artifacts.

In [160]:
from pathlib import Path
import os, json, re, math, random
import pandas as pd
from collections import Counter
from math import sqrt
from openai import OpenAI

# --- Config ---
MODEL = "ft:gpt-4o-mini-2024-07-18:personal:stance-head-v1:CXXVxeHu"
CSV_PATH = Path("./watchlist_metrics_20251102.csv")   
assert CSV_PATH.exists(), f"CSV not found: {CSV_PATH}"

# Column names 
symbol_col      = "symbol"
date_col        = None 
llm_stance_col  = "llm_stance"
llm_score_col   = "llm_score"
llm_summary_col = "llm_summary"

# System prompt (same schema as training)
SYSTEM_PROMPT = (
    """You are a disciplined technical analyst.
        Write a concise, neutral read of the chart using the provided indicators.
        - 1 to 3 sentences, no emojis or bullet points.
        - No advice or targets; describe momentum, trend, and key nearby levels if present.
        - Return STRICT JSON with keys: stance (bullish|bearish|neutral), llm_score (integer -2..2), summary (string <= 300 chars).
        - Consider RSI/MACD/Bollinger/TD/52w/support/resistance holistically."""
)

# --- Helpers (kept tiny) ---
def normalize_stance(s):
    if not isinstance(s, str): return None
    t = s.strip().lower()
    if t in {"bull","bullish"}: return "BULLISH"
    if t in {"bear","bearish"}: return "BEARISH"
    if t in {"neutral","sideways"}: return "NEUTRAL"
    return s.strip().upper()

def build_feature_cols(df: pd.DataFrame) -> list[str]:
    exclude = {symbol_col, llm_stance_col, llm_score_col, llm_summary_col}
    # exclude final decision/score columns if present
    for c in ["final_score","combined_final_score","final_decision","decision","action","recommendation"]:
        if c in df.columns: exclude.add(c)
    feats = []
    for c in df.columns:
        if c in exclude: continue
        if pd.api.types.is_numeric_dtype(df[c]):
            feats.append(c)
        else:
            # allow engineered tag-like strings
            if re.search(r"(tag|label|flag|signal|state|pattern|setup|td_last_signal)", c, flags=re.I):
                feats.append(c)
    return sorted(feats)

def row_to_features(row: pd.Series, feature_cols: list[str]) -> dict:
    def compact(x):
        try:
            return float(f"{float(x):.6g}")
        except Exception:
            return None
    feats = {"symbol": str(row[symbol_col])}
    if date_col and date_col in row.index and pd.notna(row[date_col]):
        feats["as_of"] = str(row[date_col])
    for c in feature_cols:
        v = row[c]
        if pd.api.types.is_number(v):
            vv = compact(v)
            if vv is not None: feats[c] = vv
        else:
            if pd.notna(v):
                s = str(v).strip()
                if s: feats[c] = s
    return feats

def parse_output(text: str):
    """Parse JSON first (supports {'stance':..., 'llm_score':..., 'summary':...}), then fallback to regex."""
    if not text:
        return {"stance": None, "score": None, "summary": None}

    # 1) Try JSON (including fenced code blocks)
    m = re.search(r"```(?:json)?\s*({[\s\S]*?})\s*```", text, flags=re.I)
    candidate = m.group(1) if m else text
    try:
        obj = json.loads(candidate)
        stance  = normalize_stance(obj.get("stance"))
        # accept either 'score' or 'llm_score'
        sc = obj.get("score", obj.get("llm_score"))
        try:
            score = float(sc) if sc is not None else None
        except:
            score = None
        summary = obj.get("summary")
        if stance or (score is not None) or summary:
            return {"stance": stance, "score": score, "summary": summary}
    except Exception:
        pass

    # 2) Regex fallback
    stance = None; score = None; summary = None
    m = re.search(r"stance\s*[:=\-]\s*([A-Za-z]+)", text, flags=re.I)
    if m: stance = normalize_stance(m.group(1))
    m = re.search(r"(?:llm_)?score\s*[:=\-]\s*([+-]?\d+(?:\.\d+)?)", text, flags=re.I)
    if m:
        try: score = float(m.group(1))
        except: score = None
    m = re.search(r"summary\s*[:=\-]\s*(.+)", text, flags=re.I|re.S)
    if m: summary = m.group(1).strip()
    return {"stance": stance, "score": score, "summary": summary}

def cosine_bow(a: str, b: str) -> float:
    if not a or not b: return 0.0
    tok = lambda s: re.findall(r"[a-z0-9]+", s.lower())
    ca, cb = Counter(tok(a)), Counter(tok(b))
    if not ca or not cb: return 0.0
    dot = sum(ca[t]*cb[t] for t in ca if t in cb)
    na = sqrt(sum(v*v for v in ca.values())); nb = sqrt(sum(v*v for v in cb.values()))
    return 0.0 if na==0 or nb==0 else dot/(na*nb)

# --- Load data and pick a random row ---
df = pd.read_csv(CSV_PATH)
feature_cols = build_feature_cols(df)
row = df.sample(1).iloc[0]  # random example
features = row_to_features(row, feature_cols)

# --- Call fine-tuned model ---
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your environment"
client = OpenAI()

user_text = "Derive stance, score, and a concise summary from the following engineered snapshot:\n" + json.dumps(features, ensure_ascii=False)
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":user_text}],
    temperature=0,
    max_tokens=256
)
raw = resp.choices[0].message.content or ""
pred = parse_output(raw)

# --- Reference from CSV ---
ref = {
    "stance": normalize_stance(row.get(llm_stance_col)),
    "score": float(row.get(llm_score_col)) if pd.notna(row.get(llm_score_col)) else None,
    "summary": str(row.get(llm_summary_col)) if pd.notna(row.get(llm_summary_col)) else ""
}

# --- Print compare ---
print("=== Random Example ===")
print("Symbol:", features.get("symbol"), "| As-of:", features.get("as_of"))
print("\n-- Predicted (model) --")
print(raw.strip()[:600])  # raw for transparency
print("\nParsed -> stance:", pred["stance"], "| score:", pred["score"], "\nsummary:", (pred["summary"] or "")[:300])

print("\n-- Reference (CSV) --")
print("stance:", ref["stance"], "| score:", ref["score"], "\nsummary:", ref["summary"][:300])

stance_match = ( (pred["stance"] or "").upper() == (ref["stance"] or "").upper() )
score_delta = (abs(pred["score"] - ref["score"]) if isinstance(pred["score"], (int,float)) and isinstance(ref["score"], (int,float)) else None)
summ_sim = cosine_bow(pred.get("summary") or "", ref.get("summary") or "")

print("\n-- Comparison --")
print("stance_match:", stance_match)
print("score_delta :", score_delta)
print(f"summary_similarity (cosine BOW): {summ_sim:.3f}")

=== Random Example ===
Symbol: XOM | As-of: None

-- Predicted (model) --
{"stance":"neutral","llm_score":0,"summary":"XOM shows a neutral momentum with an RSI of 58.41 and a bullish MACD. The price is near resistance at 114.69 and support at 113.8, while trading within Bollinger Bands, indicating consolidation. The TD indicator has recently signaled a sell, suggesting caution."}

Parsed -> stance: NEUTRAL | score: 0.0 
summary: XOM shows a neutral momentum with an RSI of 58.41 and a bullish MACD. The price is near resistance at 114.69 and support at 113.8, while trading within Bollinger Bands, indicating consolidation. The TD indicator has recently signaled a sell, suggesting caution.

-- Reference (CSV) --
stance: NEUTRAL | score: 0.0 
summary: XOM is currently trading near resistance at 114.69 with a close of 114.36. The RSI indicates neutral momentum at 58.41, while the MACD shows bullish momentum. The price is above support at 113.8 and inside the Bollinger Bands, suggesting a con

Looks good: the fine-tuned model returned valid JSON, my parser mapped llm_score to score, and the output aligned with the CSV. For XOM, I see stance NEUTRAL and score 0.0 matching exactly; the summary similarity = 0.796 is high given small wording differences (e.g., “shows a neutral momentum” vs “RSI indicates neutral momentum,” and phrasing around consolidation). The content covers identical drivers—RSI ~58.4, bullish MACD, near resistance 114.69, support 113.8, and inside Bollinger Bands—plus a brief TD sell caution. This tells me the stance head is reproducing my Milestone-2 LLM layer reliably and succinctly, following the schema and integer score constraint. I’m comfortable proceeding to wire the pieces into the Streamlit app.

### write_config.py — generate reusable config files
Since OpenAI fine-tuned models are hosted, I don’t “save weights” locally. What I do save (and reuse in Streamlit) is a tiny inference config: the fine-tuned model name, the system prompt, the CSV path, and the column mapping + feature list:
- a one-time script to write the config files from the current dataset,
- a small utils module I’ll import from Streamlit, and
- a complete Streamlit app with filters (stance, score range, symbol search, tag contains, etc.), table, and per-row inference/compare.

I will reuse watchlist_metrics_20251102.csv for simplicity.

#### generate reusable config files

In [177]:
# write_config.py
from pathlib import Path
import json, re
import pandas as pd

MODEL = "ft:gpt-4o-mini-2024-07-18:personal:stance-head-v1:CXXVxeHu"
CSV_PATH = Path("./watchlist_metrics_20251102.csv") # Can be updated to include the most recent data

symbol_col      = "symbol"
date_col        = None           # set to "date" if want it displayed
llm_stance_col  = "llm_stance"
llm_score_col   = "llm_score"
llm_summary_col = "llm_summary"

SYSTEM_PROMPT = (
    "You are a professional stock analyst and an equity stance assistant. "
    "Given engineered metrics for a ticker, output ONLY:\n"
    "stance: BULLISH|NEUTRAL|BEARISH\n"
    "score: <float>\n"
    "summary: <one or two short sentences>\n"
    "Do not fetch external data. Be concise and deterministic. score should be integer (-2..2)."
)

def build_feature_cols(df: pd.DataFrame) -> list[str]:
    exclude = {symbol_col, llm_stance_col, llm_score_col, llm_summary_col}
    for c in ["final_score","combined_final_score","final_decision","decision","action","recommendation"]:
        if c in df.columns: exclude.add(c)
    feats = []
    for c in df.columns:
        if c in exclude: continue
        if pd.api.types.is_numeric_dtype(df[c]):
            feats.append(c)
        else:
            if re.search(r"(tag|label|flag|signal|state|pattern|setup|td_last_signal)", c, flags=re.I):
                feats.append(c)
    return sorted(feats)

assert CSV_PATH.exists(), f"CSV not found: {CSV_PATH}"
df = pd.read_csv(CSV_PATH)
feature_cols = build_feature_cols(df)

ART = Path("./m3_artifacts"); ART.mkdir(exist_ok=True)
(ART / "last_good_prompt.txt").write_text(SYSTEM_PROMPT, encoding="utf-8")

cfg = {
    "fine_tuned_model": MODEL,
    "system_prompt_path": str((ART / "last_good_prompt.txt").resolve()),
    "csv_path": str(CSV_PATH.resolve()),
    "feature_cols": feature_cols,
    "symbol_col": symbol_col,
    "date_col": date_col
}
(ART / "inference_config.json").write_text(json.dumps(cfg, indent=2), encoding="utf-8")
print("Wrote:", (ART / "inference_config.json").resolve())
print("Feature columns:", len(feature_cols))

Wrote: /Users/ZichengXu/Desktop/DSC 670/Term Project/m3_artifacts/inference_config.json
Feature columns: 17


#### Streamlit dashboard
Filters (stance/score/symbol/tag), table view, and on-demand inference that appends predicted stance/score/summary as new columns.

In [170]:
# # app.py
# import os, json, re
# from pathlib import Path
# import pandas as pd
# import streamlit as st
# from openai import OpenAI

# st.set_page_config(page_title="Stance Head Dashboard", layout="wide")

# # --- Load config/prompt ---
# CFG_PATH = Path("./m3_artifacts/inference_config.json")
# assert CFG_PATH.exists(), "Run `python write_config.py` once to create config."
# cfg = json.loads(CFG_PATH.read_text())
# SYSTEM_PROMPT = Path(cfg["system_prompt_path"]).read_text()
# MODEL         = cfg["fine_tuned_model"]
# CSV_PATH      = Path(cfg["csv_path"])
# FEATURE_COLS  = cfg["feature_cols"]
# SYMBOL_COL    = cfg["symbol_col"]
# DATE_COL      = cfg["date_col"]  # may be None

# # --- Data ---
# @st.cache_data
# def load_df(path: Path) -> pd.DataFrame:
#     return pd.read_csv(path)
# df = load_df(CSV_PATH)

# # --- Helpers (minimal) ---
# def normalize_stance(s):
#     if not isinstance(s, str): return None
#     t = s.strip().lower()
#     if t in {"bull","bullish"}: return "BULLISH"
#     if t in {"bear","bearish"}: return "BEARISH"
#     if t in {"neutral","sideways"}: return "NEUTRAL"
#     return s.strip().upper()

# def row_to_features(row: pd.Series, feature_cols: list[str], symbol_col: str, date_col: str | None):
#     def compact(x):
#         try: return float(f"{float(x):.6g}")
#         except: return None
#     feats = {"symbol": str(row[symbol_col])}
#     if date_col and date_col in row.index and pd.notna(row[date_col]):
#         feats["as_of"] = str(row[date_col])
#     for c in feature_cols:
#         v = row[c]
#         if pd.api.types.is_number(v):
#             vv = compact(v)
#             if vv is not None: feats[c] = vv
#         else:
#             if pd.notna(v):
#                 s = str(v).strip()
#                 if s: feats[c] = s
#     return feats

# def parse_output(text: str):
#     if not text: return {"stance": None, "score": None, "summary": None}
#     m = re.search(r"```(?:json)?\s*({[\s\S]*?})\s*```", text, flags=re.I)
#     candidate = m.group(1) if m else text
#     try:
#         obj = json.loads(candidate)
#         stance = normalize_stance(obj.get("stance"))
#         sc = obj.get("score", obj.get("llm_score"))
#         score = float(sc) if sc is not None else None
#         summary = obj.get("summary")
#         return {"stance": stance, "score": score, "summary": summary}
#     except Exception:
#         pass
#     # fallback regex
#     stance = None; score = None; summary = None
#     m = re.search(r"stance\s*[:=\-]\s*([A-Za-z]+)", candidate, flags=re.I)
#     if m: stance = normalize_stance(m.group(1))
#     m = re.search(r"(?:llm_)?score\s*[:=\-]\s*([+-]?\d+(?:\.\d+)?)", candidate, flags=re.I)
#     if m:
#         try: score = float(m.group(1))
#         except: score = None
#     m = re.search(r"summary\s*[:=\-]\s*(.+)", candidate, flags=re.I|re.S)
#     if m: summary = m.group(1).strip()
#     return {"stance": stance, "score": score, "summary": summary}

# def run_inference_row(row: pd.Series):
#     assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY"
#     client = OpenAI()
#     feats = row_to_features(row, FEATURE_COLS, SYMBOL_COL, DATE_COL)
#     user_text = "Derive stance, score, and a concise summary from the following engineered snapshot:\n" + json.dumps(feats, ensure_ascii=False)
#     resp = client.chat.completions.create(
#         model=MODEL,
#         messages=[{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":user_text}],
#         temperature=0,
#         max_tokens=256
#     )
#     raw = resp.choices[0].message.content or ""
#     return parse_output(raw), feats, raw

# # --- UI: Filters ---
# st.title("Stance Head Dashboard (Fine-tuned)")

# with st.sidebar:
#     st.header("Filters")
#     # Symbol filter
#     symbols = sorted(df[SYMBOL_COL].dropna().unique().tolist())
#     sym_sel = st.multiselect("Symbols", symbols, default=[])
#     # stance (from your CSV column if present)
#     stance_col = None
#     for c in df.columns:
#         if c.lower() == "llm_stance": stance_col = c; break
#     stance_opts = ["BULLISH","NEUTRAL","BEARISH"]
#     stance_sel = st.multiselect("Stance (CSV)", stance_opts, default=stance_opts) if stance_col else []
#     # score range (CSV)
#     score_col = None
#     for c in df.columns:
#         if c.lower() == "llm_score": score_col = c; break
#     if score_col:
#         smin, smax = float(df[score_col].min()), float(df[score_col].max())
#         score_rng = st.slider("Score range (CSV)", -2.0, 2.0, (smin, smax))
#     else:
#         score_rng = (-2.0, 2.0)
#     # tag contains
#     tag_cols = [c for c in df.columns if "tag" in c.lower()]
#     tag_field = tag_cols[0] if tag_cols else None
#     tag_query = st.text_input(f"Tag contains ({tag_field})", "")

# # Apply filters
# view = df.copy()
# if sym_sel: view = view[view[SYMBOL_COL].isin(sym_sel)]
# if stance_col and stance_sel:
#     view = view[view[stance_col].apply(lambda s: normalize_stance(s) in set(stance_sel))]
# if score_col:
#     view = view[(view[score_col] >= score_rng[0]) & (view[score_col] <= score_rng[1])]
# if tag_field and tag_query.strip():
#     view = view[view[tag_field].astype(str).str.contains(tag_query.strip(), case=False, na=False)]

# st.write(f"**Rows:** {len(view)}")
# st.dataframe(view, use_container_width=True, height=480)

# st.divider()
# st.subheader("Run Inference")

# left, right = st.columns([1,1])
# with left:
#     idx = st.number_input("Row index (from filtered table above)", min_value=0, max_value=max(0, len(view)-1), value=0)
# with right:
#     run_btn = st.button("Predict stance/score/summary")

# if run_btn:
#     row = view.iloc[int(idx)]
#     pred, feats, raw = run_inference_row(row)
#     st.markdown("### Features sent to model")
#     st.json(feats)
#     st.markdown("### Model raw output")
#     st.code(raw)
#     st.markdown("### Parsed")
#     st.json(pred)

#     # append to the shown table (in-memory)
#     view = view.copy()
#     view.loc[view.index[int(idx)], "pred_stance"]  = pred["stance"]
#     view.loc[view.index[int(idx)], "pred_score"]   = pred["score"]
#     view.loc[view.index[int(idx)], "pred_summary"] = pred["summary"]
#     st.markdown("### Updated table (with prediction columns)")
#     st.dataframe(view, use_container_width=True, height=480)

# st.caption(f"Model: {MODEL}")

In [173]:
# # app.py
# import streamlit as st
# import pandas as pd
# from pathlib import Path
# from stance_utils import (
#     load_config, row_to_features, run_inference_openai, normalize_stance, cosine_bow
# )

# st.set_page_config(page_title="Stance Head Dashboard", layout="wide")

# # --- Load config & data ---
# CFG_PATH = Path("./m3_artifacts/inference_config.json")
# assert CFG_PATH.exists(), "Run `python write_config.py` first to create m3_artifacts/inference_config.json"
# cfg = load_config(CFG_PATH)

# MODEL            = cfg["fine_tuned_model"]
# SYSTEM_PROMPT    = cfg["system_prompt"]
# CSV_PATH         = Path(cfg["csv_path"])
# FEATURE_COLS     = cfg["feature_cols"]
# SYMBOL_COL       = cfg["symbol_col"]
# DATE_COL         = cfg["date_col"]
# LLM_STANCE_COL   = cfg["llm_stance_col"]
# LLM_SCORE_COL    = cfg["llm_score_col"]
# LLM_SUMMARY_COL  = cfg["llm_summary_col"]

# @st.cache_data
# def load_df(path: Path) -> pd.DataFrame:
#     df = pd.read_csv(path)
#     # normalize stance column to canonical set for filtering
#     if LLM_STANCE_COL in df.columns:
#         df["_stance_norm"] = df[LLM_STANCE_COL].apply(normalize_stance)
#     else:
#         df["_stance_norm"] = None
#     return df

# df = load_df(CSV_PATH)

# st.title("Stance Head Dashboard (Fine-tuned)")

# with st.sidebar:
#     st.header("Filters")
#     # symbol search / multiselect
#     symbols = sorted(df[SYMBOL_COL].dropna().unique().tolist())
#     sym_sel = st.multiselect("Symbols", symbols, default=[])
#     # stance filter
#     stance_options = ["BULLISH","NEUTRAL","BEARISH"]
#     stance_sel = st.multiselect("LLM stance (from CSV)", stance_options, default=stance_options)
#     # score range
#     min_sc = float(df[LLM_SCORE_COL].min()) if LLM_SCORE_COL in df.columns else -2.0
#     max_sc = float(df[LLM_SCORE_COL].max()) if LLM_SCORE_COL in df.columns else 2.0
#     score_range = st.slider("LLM score range (from CSV)", -2.0, 2.0, (min_sc, max_sc))
#     # tag contains
#     tag_cols = [c for c in df.columns if "tag" in c.lower()]
#     tags_field = tag_cols[0] if tag_cols else None
#     tag_query = st.text_input(f"Tag contains ({tags_field})", value="")

#     st.divider()
#     st.subheader("Model")
#     st.code(MODEL, language="text")
#     st.caption("Make sure OPENAI_API_KEY is present in the environment before running inference.")

# # --- Apply filters ---
# view = df.copy()
# if sym_sel:
#     view = view[view[SYMBOL_COL].isin(sym_sel)]
# if stance_sel:
#     view = view[view["_stance_norm"].isin(stance_sel)]
# if LLM_SCORE_COL in view.columns:
#     view = view[(view[LLM_SCORE_COL] >= score_range[0]) & (view[LLM_SCORE_COL] <= score_range[1])]
# if tags_field and tag_query.strip():
#     view = view[view[tags_field].astype(str).str.contains(tag_query.strip(), case=False, na=False)]

# st.write(f"**Rows:** {len(view)}")
# st.dataframe(view[[SYMBOL_COL, LLM_STANCE_COL, LLM_SCORE_COL, LLM_SUMMARY_COL] + [c for c in view.columns if c not in {SYMBOL_COL, LLM_STANCE_COL, LLM_SCORE_COL, LLM_SUMMARY_COL, '_stance_norm'}]].head(2000), use_container_width=True)

# st.divider()
# st.subheader("Per-row Inference")

# col1, col2, col3 = st.columns([1,1,1])
# with col1:
#     row_idx = st.number_input("Row index (from filtered table)", min_value=0, max_value=max(0, len(view)-1), value=0, step=1)
# with col2:
#     temp = st.slider("temperature", 0.0, 1.0, 0.0, 0.1)
# with col3:
#     max_tokens = st.slider("max_tokens", 64, 512, 256, 32)

# if st.button("Run inference on selected row"):
#     row = view.iloc[int(row_idx)]
#     feats = row_to_features(row, FEATURE_COLS, SYMBOL_COL, DATE_COL)
#     out = run_inference_openai(feats, MODEL, SYSTEM_PROMPT, temperature=temp, max_tokens=max_tokens)

#     pred = out["parsed"]
#     raw  = out["raw"]
#     ref_stance  = normalize_stance(row.get(LLM_STANCE_COL))
#     ref_score   = float(row.get(LLM_SCORE_COL)) if pd.notna(row.get(LLM_SCORE_COL)) else None
#     ref_summary = (row.get(LLM_SUMMARY_COL) or "") if pd.notna(row.get(LLM_SUMMARY_COL)) else ""

#     stance_match = (pred.get("stance") or "").upper() == (ref_stance or "").upper()
#     score_delta  = abs(pred.get("score") - ref_score) if isinstance(pred.get("score"), (int,float)) and isinstance(ref_score, (int,float)) else None
#     summ_sim     = cosine_bow(pred.get("summary") or "", ref_summary or "")

#     st.markdown("### Selected features")
#     st.json(feats)

#     st.markdown("### Model output (raw)")
#     st.code(raw)

#     st.markdown("### Parsed")
#     st.json(pred)

#     st.markdown("### Reference (from CSV)")
#     st.json({"stance": ref_stance, "score": ref_score, "summary": (ref_summary or "")[:500]})

#     st.markdown("### Comparison")
#     st.write({
#         "stance_match": stance_match,
#         "score_delta": score_delta,
#         "summary_similarity_bow": round(summ_sim, 3)
#     })

# st.divider()
# st.subheader("Batch Inference (current filtered view)")

# nmax = st.slider("How many rows to score (head of filtered view)?", 1, max(1, len(view)), min(50, len(view)))
# if st.button("Run batch inference"):
#     rows = []
#     sub = view.head(nmax).copy()
#     for _, r in sub.iterrows():
#         feats = row_to_features(r, FEATURE_COLS, SYMBOL_COL, DATE_COL)
#         out   = run_inference_openai(feats, MODEL, SYSTEM_PROMPT, temperature=0, max_tokens=256)
#         pred  = out["parsed"]
#         rows.append({
#             "symbol": r[SYMBOL_COL],
#             "pred_stance": pred.get("stance"),
#             "pred_score": pred.get("score"),
#             "pred_summary": pred.get("summary"),
#             "ref_stance": normalize_stance(r.get(LLM_STANCE_COL)),
#             "ref_score": r.get(LLM_SCORE_COL),
#             "ref_summary": r.get(LLM_SUMMARY_COL)
#         })
#     res = pd.DataFrame(rows)
#     st.dataframe(res, use_container_width=True)
#     st.download_button("Download results CSV", res.to_csv(index=False).encode("utf-8"), file_name="stance_head_results.csv", mime="text/csv")

## Streamlit App Building